# 06c: Information-Theoretic Analysis -- Circuit Mean Ablation

## Overview
This notebook quantifies **how much information about frequency band identity**
is preserved in model activations under **mean ablation** (circuit mode), using
information-theoretic measures. Non-circuit edges have their outputs replaced
with dataset-mean activations, isolating the computational path through the
ACDC-discovered circuit.

## Key Questions
- How many bits of band information survive circuit mean ablation at each layer?
- Which layers lose the most band information under ablation?
- Do attention and MLP components in the circuit carry different amounts of band MI?
- How does circuit coding efficiency compare to the full model?
- Do MI-based and geometric measures (CKA from NB02c) agree on circuit impact?

## Hypothesis Domain: R5-Circuit
- **H-R5c.1**: Circuit MI trajectory is lower but shape-preserving relative to base
- **H-R5c.2**: Information loss concentrates in mid-to-late layers where ablation is strongest
- **H-R5c.3**: MLP components in-circuit carry more residual band MI than attention
- **H-R5c.4**: Coding efficiency (bits/dim) decreases under ablation
- **H-R5c.5**: MI-based information loss correlates with CKA degradation from NB02c

## Notebook Structure
1. Setup and Data Loading
2. Circuit MI Trajectory (KSG + probe-based)
3. Circuit Conditional Entropy
4. Circuit Delta-MI (per-layer information gain)
5. Circuit Component MI (attn_out vs mlp_out)
6. Circuit Coding Efficiency
7. Base vs Circuit Comparison
8. Integration with Geometric Measures (CKA from NB02c)
9. Summary

## Data Sources
- Circuit activations: `outputs/extraction/circuit_activations/`
- Base info-theoretic results: `outputs/info_theoretic/base/analysis/06_*.csv`
- Base-circuit CKA: `outputs/residual_stream/comparison/analysis/02c_base_circuit_cka.csv`

## 1. Setup and Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from functools import partial as _partial
from sklearn.decomposition import PCA

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    K_NEIGHBORS,
    RANDOM_SEED,
    get_domain_dirs,
)
from utils.data_loading import save_analysis, load_domain_csv
from utils.circuit_loading import (
    load_circuit_activations,
    load_base_and_circuit,
)
from utils.info_theory import (
    estimate_mi_ksg,
    probe_based_mi,
    compute_conditional_entropy,
    compute_mi_trajectory,
    compute_delta_mi,
    compute_coding_efficiency,
    compute_component_mi,
)
from utils.probing import train_probe
from utils.plotting import setup_plotting, save_figure, plot_mi_trajectory

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()

CIRCUIT_ANALYSIS, CIRCUIT_VIZ = get_domain_dirs("info_theoretic", "circuit")
COMP_ANALYSIS, COMP_VIZ = get_domain_dirs("info_theoretic", "comparison")
save_analysis_circuit = _partial(save_analysis, analysis_dir=CIRCUIT_ANALYSIS)
save_figure_circuit = _partial(save_figure, viz_dir=CIRCUIT_VIZ)
save_analysis_comp = _partial(save_analysis, analysis_dir=COMP_ANALYSIS)
save_figure_comp = _partial(save_figure, viz_dir=COMP_VIZ)

print(f"Models: {MODELS}")
print(f"Bands:  {BANDS}")
print(f"Draws:  {DRAWS}")
print(f"K_NEIGHBORS: {K_NEIGHBORS}")
print(f"Circuit analysis dir: {CIRCUIT_ANALYSIS}")
print(f"Circuit viz dir:      {CIRCUIT_VIZ}")
print(f"Comparison analysis:  {COMP_ANALYSIS}")
print(f"Comparison viz:       {COMP_VIZ}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands:  ['low', 'medium', 'high', 'very_high', 'control']
Draws:  ['draw_1', 'draw_2', 'draw_3']
K_NEIGHBORS: 10
Circuit analysis dir: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/circuit/analysis
Circuit viz dir:      LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/circuit/viz
Comparison analysis:  LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/comparison/analysis
Comparison viz:       LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/comparison/viz


In [2]:
# Load circuit activations for all models/bands/draws
# resid_post_predpos: (N, n_layers, d_model)
# attn_out_predpos:   (N, n_layers, d_model)
# mlp_out_predpos:    (N, n_layers, d_model)

all_circuit_resid = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_circuit_attn = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_circuit_mlp = {}  # model -> draw -> {band: (N, n_layers, d_model)}

for model in MODELS:
    all_circuit_resid[model] = {}
    all_circuit_attn[model] = {}
    all_circuit_mlp[model] = {}
    for draw in DRAWS:
        all_circuit_resid[model][draw] = {}
        all_circuit_attn[model][draw] = {}
        all_circuit_mlp[model][draw] = {}
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
                all_circuit_resid[model][draw][band] = data["resid_post_predpos"]
                if "attn_out_predpos" in data:
                    all_circuit_attn[model][draw][band] = data["attn_out_predpos"]
                if "mlp_out_predpos" in data:
                    all_circuit_mlp[model][draw][band] = data["mlp_out_predpos"]
            except FileNotFoundError:
                pass

# Report shapes
for model in MODELS:
    sample = next(iter(next(iter(all_circuit_resid[model].values())).values()), None)
    if sample is not None:
        print(f"{model}: circuit resid shape = {sample.shape}  (N, n_layers, d_model)")
    attn_sample = next(
        iter(next(iter(all_circuit_attn[model].values())).values()), None
    )
    if attn_sample is not None:
        print(f"  circuit attn_out shape = {attn_sample.shape}")
    mlp_sample = next(iter(next(iter(all_circuit_mlp[model].values())).values()), None)
    if mlp_sample is not None:
        print(f"  circuit mlp_out shape  = {mlp_sample.shape}")

pythia-70m: circuit resid shape = (225, 6, 512)  (N, n_layers, d_model)
  circuit attn_out shape = (225, 6, 512)
  circuit mlp_out shape  = (225, 6, 512)
pythia-160m: circuit resid shape = (225, 12, 768)  (N, n_layers, d_model)
  circuit attn_out shape = (225, 12, 768)
  circuit mlp_out shape  = (225, 12, 768)
pythia-410m: circuit resid shape = (225, 24, 1024)  (N, n_layers, d_model)
  circuit attn_out shape = (225, 24, 1024)
  circuit mlp_out shape  = (225, 24, 1024)
pythia-1b: circuit resid shape = (225, 16, 2048)  (N, n_layers, d_model)
  circuit attn_out shape = (225, 16, 2048)
  circuit mlp_out shape  = (225, 16, 2048)
pythia-1.4b: circuit resid shape = (225, 24, 2048)  (N, n_layers, d_model)
  circuit attn_out shape = (225, 24, 2048)
  circuit mlp_out shape  = (225, 24, 2048)


In [3]:
# Helper functions


def build_circuit_layer_data(model, draw, layer, act_dict=None):
    """Combine all bands into (X, labels) for a specific layer under circuit mode.

    Args:
        model: Model name.
        draw: Draw name.
        layer: Layer index.
        act_dict: Activation dict (default: all_circuit_resid).

    Returns:
        (X, labels) where X is (N_total, d_model) and labels is (N_total,),
        or (None, None) if insufficient data.
    """
    if act_dict is None:
        act_dict = all_circuit_resid
    embs, labels = [], []
    for band in BANDS:
        act = act_dict.get(model, {}).get(draw, {}).get(band)
        if act is not None:
            embs.append(act[:, layer, :])
            labels.extend([band] * act.shape[0])
    if len(embs) < 2:
        return None, None
    return np.vstack(embs), np.array(labels)


def reduce_dims(X, n_components=10):
    """Apply PCA dimensionality reduction before KSG MI estimation."""
    if X.shape[1] <= n_components:
        return X
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    return pca.fit_transform(X)


n_classes = len(BANDS)
H_Y = np.log2(n_classes)  # Max entropy assuming uniform prior
print(f"Number of classes: {n_classes}")
print(f"H(Y) = log2({n_classes}) = {H_Y:.3f} bits (uniform prior upper bound)")

Number of classes: 5
H(Y) = log2(5) = 2.322 bits (uniform prior upper bound)


## 2. Circuit MI Trajectory

MI(circuit_resid; band) per layer using:
1. **KSG estimator**: non-parametric, based on k-NN distances (PCA to 10 dims first)
2. **Probe-based MI**: lower bound from linear probe predictions

This measures how much frequency band information the circuit-constrained
residual stream carries at each processing stage.

In [4]:
# KSG MI estimation on circuit residual stream at each layer
mi_ksg_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_model = MODEL_D_MODEL[model]
    print(f"\n{model} ({n_layers} layers, d_model={d_model}):")

    for draw in DRAWS:
        for layer in range(n_layers):
            X, labels = build_circuit_layer_data(model, draw, layer)
            if X is None:
                continue

            # PCA to 10 dims before KSG
            X_reduced = reduce_dims(X, n_components=10)

            mi = estimate_mi_ksg(X_reduced, labels, k=K_NEIGHBORS)

            mi_ksg_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "mi_ksg": mi,
                    "method": "ksg",
                    "d_model": d_model,
                }
            )

        # Progress report (draw_1 only)
        if draw == "draw_1":
            draw_records = [
                r for r in mi_ksg_records if r["model"] == model and r["draw"] == draw
            ]
            if draw_records:
                peak = max(draw_records, key=lambda r: r["mi_ksg"])
                print(
                    f"  {draw}: peak circuit MI(KSG) = {peak['mi_ksg']:.3f} bits "
                    f"at layer {peak['layer']}"
                )

df_circuit_mi_ksg = pd.DataFrame(mi_ksg_records)
print(f"\nCircuit KSG MI records: {len(df_circuit_mi_ksg)}")


pythia-70m (6 layers, d_model=512):


  draw_1: peak circuit MI(KSG) = 0.759 bits at layer 0



pythia-160m (12 layers, d_model=768):


  draw_1: peak circuit MI(KSG) = 0.899 bits at layer 6



pythia-410m (24 layers, d_model=1024):


...


pythia-1b (16 layers, d_model=2048):


  draw_1: peak circuit MI(KSG) = 1.225 bits at layer 13



pythia-1.4b (24 layers, d_model=2048):


  draw_1: peak circuit MI(KSG) = 1.200 bits at layer 22



Circuit KSG MI records: 246


In [5]:
# Probe-based MI estimation on circuit residual stream
mi_probe_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model} ({n_layers} layers):")

    # Key layers: every other layer for large models
    if n_layers <= 8:
        probe_layers = list(range(n_layers))
    else:
        probe_layers = sorted(set([0] + list(range(0, n_layers, 2)) + [n_layers - 1]))

    for draw in ["draw_1"]:
        for layer in probe_layers:
            X, labels = build_circuit_layer_data(model, draw, layer)
            if X is None:
                continue

            # Train linear probe and get predictions
            probe_result = train_probe(X, labels, return_predictions=True)
            predictions = probe_result["predictions"]
            true_labels = probe_result["true_labels"]

            # Probe-based MI from confusion matrix
            mi_probe = probe_based_mi(predictions, true_labels)

            mi_probe_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "mi_probe": mi_probe,
                    "probe_accuracy": probe_result["accuracy"],
                    "method": "probe",
                }
            )

        draw_records = [
            r for r in mi_probe_records if r["model"] == model and r["draw"] == draw
        ]
        if draw_records:
            peak = max(draw_records, key=lambda r: r["mi_probe"])
            print(
                f"  {draw}: peak circuit MI(probe) = {peak['mi_probe']:.3f} bits "
                f"at layer {peak['layer']}"
            )

df_circuit_mi_probe = pd.DataFrame(mi_probe_records)
print(f"\nCircuit probe-based MI records: {len(df_circuit_mi_probe)}")


pythia-70m (6 layers):


  draw_1: peak circuit MI(probe) = 0.946 bits at layer 3

pythia-160m (12 layers):


  draw_1: peak circuit MI(probe) = 1.144 bits at layer 6

pythia-410m (24 layers):


  draw_1: peak circuit MI(probe) = 1.271 bits at layer 20

pythia-1b (16 layers):


  draw_1: peak circuit MI(probe) = 1.484 bits at layer 15

pythia-1.4b (24 layers):


  draw_1: peak circuit MI(probe) = 1.541 bits at layer 22

Circuit probe-based MI records: 48


In [6]:
# Combine KSG and probe MI into a unified trajectory DataFrame
df_circuit_mi_combined = df_circuit_mi_ksg[
    ["model", "draw", "layer", "mi_ksg", "d_model"]
].merge(
    df_circuit_mi_probe[["model", "draw", "layer", "mi_probe", "probe_accuracy"]],
    on=["model", "draw", "layer"],
    how="outer",
)

save_analysis_circuit(df_circuit_mi_combined, "06c_circuit_mi_trajectory.csv")
print(f"Combined circuit MI records: {len(df_circuit_mi_combined)}")
print(df_circuit_mi_combined.head(10))

Combined circuit MI records: 246
         model    draw  layer    mi_ksg  d_model  mi_probe  probe_accuracy
0  pythia-1.4b  draw_1      0  0.995143     2048  0.895508        0.591111
1  pythia-1.4b  draw_1      1  1.054054     2048       NaN             NaN
2  pythia-1.4b  draw_1      2  1.021862     2048  1.062137        0.680889
3  pythia-1.4b  draw_1      3  0.998386     2048       NaN             NaN
4  pythia-1.4b  draw_1      4  0.984216     2048  1.051691        0.662222
5  pythia-1.4b  draw_1      5  0.949061     2048       NaN             NaN
6  pythia-1.4b  draw_1      6  0.980519     2048  1.031679        0.664000
7  pythia-1.4b  draw_1      7  0.868883     2048       NaN             NaN
8  pythia-1.4b  draw_1      8  0.880496     2048  1.233847        0.728000
9  pythia-1.4b  draw_1      9  0.840659     2048       NaN             NaN


### Visualization: Circuit MI Trajectories

In [7]:
# Plot circuit MI trajectories per model: KSG vs probe-based
for model in MODELS:
    model_data = df_circuit_mi_combined[
        (df_circuit_mi_combined["model"] == model)
        & (df_circuit_mi_combined["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(
        model_data["layer"],
        model_data["mi_ksg"],
        color="#1f77b4",
        label="MI (KSG)",
        marker="o",
        markersize=4,
    )
    if model_data["mi_probe"].notna().any():
        probe_data = model_data.dropna(subset=["mi_probe"])
        ax.plot(
            probe_data["layer"],
            probe_data["mi_probe"],
            color="#ff7f0e",
            label="MI (Probe-based)",
            marker="s",
            markersize=4,
        )

    ax.axhline(
        y=H_Y, color="gray", linestyle=":", alpha=0.5, label=f"H(Y)={H_Y:.2f} bits"
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mutual Information (bits)")
    ax.set_title(f"Circuit MI(resid; band) Trajectory -- {model}")
    ax.legend()
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_06c_01_circuit_mi_trajectory_{model}.png")

In [8]:
# All models on one plot (circuit KSG MI, draw_1, normalized depth)
fig, ax = plt.subplots(figsize=(14, 7))
for model in MODELS:
    model_data = df_circuit_mi_combined[
        (df_circuit_mi_combined["model"] == model)
        & (df_circuit_mi_combined["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    n_layers = MODEL_INFO[model]["n_layers"]
    layer_frac = model_data["layer"] / max(n_layers - 1, 1)
    ax.plot(
        layer_frac,
        model_data["mi_ksg"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

ax.axhline(y=H_Y, color="gray", linestyle=":", alpha=0.5, label=f"H(Y)={H_Y:.2f}")
ax.set_xlabel("Relative Layer Position (0=first, 1=last)")
ax.set_ylabel("MI (KSG, bits)")
ax.set_title("Circuit MI(resid; band) Across Models (Normalized Depth)")
ax.legend()
ax.set_ylim(bottom=0)
fig.tight_layout()
save_figure_circuit(fig, "viz_06c_02_circuit_mi_all_models.png")

## 3. Circuit Conditional Entropy

H(band | circuit_resid) = H(band) - MI(circuit_resid; band) per layer.
Measures the remaining uncertainty about band identity after observing
the circuit-constrained residual stream.

In [9]:
cond_ent_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model}:")

    for draw in DRAWS:
        for layer in range(n_layers):
            X, labels = build_circuit_layer_data(model, draw, layer)
            if X is None:
                continue

            X_reduced = reduce_dims(X, n_components=10)
            h_cond = compute_conditional_entropy(X_reduced, labels, k=K_NEIGHBORS)

            # Marginal H(Y) from label distribution
            unique, counts = np.unique(labels, return_counts=True)
            p_y = counts / len(labels)
            h_y = -np.sum(p_y * np.log2(np.clip(p_y, 1e-10, 1.0)))

            cond_ent_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "h_y": h_y,
                    "h_y_given_x": h_cond,
                    "info_fraction": 1.0 - (h_cond / h_y) if h_y > 0 else 0.0,
                }
            )

        if draw == "draw_1":
            draw_recs = [
                r for r in cond_ent_records if r["model"] == model and r["draw"] == draw
            ]
            if draw_recs:
                best = min(draw_recs, key=lambda r: r["h_y_given_x"])
                print(
                    f"  {draw}: min H(Y|X_circuit) = {best['h_y_given_x']:.3f} bits "
                    f"at layer {best['layer']}"
                )
                print(f"          info fraction = {best['info_fraction']:.3f}")

df_circuit_cond_ent = pd.DataFrame(cond_ent_records)
save_analysis_circuit(df_circuit_cond_ent, "06c_circuit_conditional_entropy.csv")
print(f"\nCircuit conditional entropy records: {len(df_circuit_cond_ent)}")


pythia-70m:


  draw_1: min H(Y|X_circuit) = 1.563 bits at layer 0
          info fraction = 0.327



pythia-160m:


  draw_1: min H(Y|X_circuit) = 1.423 bits at layer 6
          info fraction = 0.387



pythia-410m:


...


pythia-1b:


  draw_1: min H(Y|X_circuit) = 1.097 bits at layer 13
          info fraction = 0.527



pythia-1.4b:


  draw_1: min H(Y|X_circuit) = 1.122 bits at layer 22
          info fraction = 0.517



Circuit conditional entropy records: 246


In [10]:
# Plot circuit conditional entropy trajectory
for model in MODELS:
    model_data = df_circuit_cond_ent[
        (df_circuit_cond_ent["model"] == model)
        & (df_circuit_cond_ent["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: H(Y|X)
    axes[0].plot(
        model_data["layer"],
        model_data["h_y_given_x"],
        color="#d62728",
        marker="o",
        markersize=4,
    )
    axes[0].axhline(
        y=model_data["h_y"].iloc[0],
        color="gray",
        linestyle=":",
        alpha=0.5,
        label=f"H(Y) = {model_data['h_y'].iloc[0]:.2f}",
    )
    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("H(band | circuit_resid) (bits)")
    axes[0].set_title(f"Circuit Conditional Entropy -- {model}")
    axes[0].legend()
    axes[0].set_ylim(bottom=0)

    # Right: Information fraction
    axes[1].plot(
        model_data["layer"],
        model_data["info_fraction"],
        color="#9467bd",
        marker="o",
        markersize=4,
    )
    axes[1].axhline(y=1.0, color="gray", linestyle=":", alpha=0.3)
    axes[1].set_xlabel("Layer")
    axes[1].set_ylabel("Info Fraction (1 - H(Y|X)/H(Y))")
    axes[1].set_title(f"Circuit Band Entropy Captured -- {model}")
    axes[1].set_ylim(-0.05, 1.05)

    fig.tight_layout()
    save_figure_circuit(fig, f"viz_06c_03_circuit_cond_entropy_{model}.png")

## 4. Circuit Delta-MI

Per-layer information gain under circuit ablation:
delta_MI(L) = MI(L) - MI(L-1).
Identifies which layers in the circuit add vs lose band information.

In [11]:
delta_mi_records = []

for model in MODELS:
    for draw in DRAWS:
        model_draw_mi = df_circuit_mi_combined[
            (df_circuit_mi_combined["model"] == model)
            & (df_circuit_mi_combined["draw"] == draw)
        ].sort_values("layer")

        if len(model_draw_mi) < 2:
            continue

        # Compute delta-MI from KSG trajectory
        mi_traj = [
            {"layer": row["layer"], "mi": row["mi_ksg"]}
            for _, row in model_draw_mi.iterrows()
            if not np.isnan(row["mi_ksg"])
        ]
        if len(mi_traj) < 2:
            continue

        delta_results = compute_delta_mi(mi_traj)
        for entry in delta_results:
            delta_mi_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": entry["layer"],
                    "delta_mi": entry["delta_mi"],
                    "cumulative_mi": entry["cumulative_mi"],
                }
            )

df_circuit_delta_mi = pd.DataFrame(delta_mi_records)
save_analysis_circuit(df_circuit_delta_mi, "06c_circuit_delta_mi.csv")
print(f"Circuit delta-MI records: {len(df_circuit_delta_mi)}")

# Report peak delta-MI layers
for model in MODELS:
    model_data = df_circuit_delta_mi[
        (df_circuit_delta_mi["model"] == model)
        & (df_circuit_delta_mi["draw"] == "draw_1")
    ]
    if len(model_data) > 0:
        peak = model_data.loc[model_data["delta_mi"].idxmax()]
        n_layers = MODEL_INFO[model]["n_layers"]
        print(
            f"{model}: peak circuit delta-MI = {peak['delta_mi']:.4f} bits "
            f"at layer {int(peak['layer'])} "
            f"({peak['layer'] / max(n_layers - 1, 1):.1%} depth)"
        )

Circuit delta-MI records: 246
pythia-70m: peak circuit delta-MI = 0.7587 bits at layer 0 (0.0% depth)
pythia-160m: peak circuit delta-MI = 0.8461 bits at layer 0 (0.0% depth)
pythia-410m: peak circuit delta-MI = 0.8612 bits at layer 0 (0.0% depth)
pythia-1b: peak circuit delta-MI = 1.0931 bits at layer 0 (0.0% depth)
pythia-1.4b: peak circuit delta-MI = 0.9951 bits at layer 0 (0.0% depth)


In [12]:
# Plot circuit delta-MI per model
for model in MODELS:
    model_data = df_circuit_delta_mi[
        (df_circuit_delta_mi["model"] == model)
        & (df_circuit_delta_mi["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: delta-MI bar chart
    colors = ["#2ca02c" if v >= 0 else "#d62728" for v in model_data["delta_mi"]]
    axes[0].bar(model_data["layer"], model_data["delta_mi"], color=colors, alpha=0.8)
    axes[0].axhline(y=0, color="black", linewidth=0.5)
    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("Delta MI (bits)")
    axes[0].set_title(f"Circuit Per-Layer Information Gain -- {model}")

    # Right: cumulative MI
    axes[1].plot(
        model_data["layer"],
        model_data["cumulative_mi"],
        color="#1f77b4",
        marker="o",
        markersize=4,
    )
    axes[1].axhline(
        y=H_Y, color="gray", linestyle=":", alpha=0.5, label=f"H(Y)={H_Y:.2f}"
    )
    axes[1].set_xlabel("Layer")
    axes[1].set_ylabel("Cumulative MI (bits)")
    axes[1].set_title(f"Circuit Cumulative MI -- {model}")
    axes[1].legend()
    axes[1].set_ylim(bottom=0)

    fig.tight_layout()
    save_figure_circuit(fig, f"viz_06c_04_circuit_delta_mi_{model}.png")

## 5. Circuit Component MI

Separate MI estimates for attention outputs vs MLP outputs under circuit
mean ablation at each layer. Reveals which component type retains more
band information when non-circuit edges are ablated.

In [13]:
component_mi_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model}:")

    for draw in DRAWS:
        # Check if we have circuit component data
        has_attn = any(
            all_circuit_attn.get(model, {}).get(draw, {}).get(b) is not None
            for b in BANDS
        )
        has_mlp = any(
            all_circuit_mlp.get(model, {}).get(draw, {}).get(b) is not None
            for b in BANDS
        )

        if not (has_attn and has_mlp):
            print(f"  {draw}: Missing circuit attn_out or mlp_out -- skipping")
            continue

        for layer in range(n_layers):
            # Circuit attention outputs
            X_attn, labels_attn = build_circuit_layer_data(
                model, draw, layer, act_dict=all_circuit_attn
            )
            # Circuit MLP outputs
            X_mlp, labels_mlp = build_circuit_layer_data(
                model, draw, layer, act_dict=all_circuit_mlp
            )

            if X_attn is None or X_mlp is None:
                continue

            # PCA to 10 dims before KSG
            X_attn_r = reduce_dims(X_attn, n_components=10)
            X_mlp_r = reduce_dims(X_mlp, n_components=10)

            attn_mi = estimate_mi_ksg(X_attn_r, labels_attn, k=K_NEIGHBORS)
            mlp_mi = estimate_mi_ksg(X_mlp_r, labels_mlp, k=K_NEIGHBORS)

            component_mi_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "attn_mi": attn_mi,
                    "mlp_mi": mlp_mi,
                    "mlp_advantage": mlp_mi - attn_mi,
                }
            )

        if draw == "draw_1":
            draw_recs = [
                r
                for r in component_mi_records
                if r["model"] == model and r["draw"] == draw
            ]
            if draw_recs:
                peak_attn = max(draw_recs, key=lambda r: r["attn_mi"])
                peak_mlp = max(draw_recs, key=lambda r: r["mlp_mi"])
                print(
                    f"  {draw}: peak circuit attn MI = {peak_attn['attn_mi']:.3f} "
                    f"(L{peak_attn['layer']})"
                )
                print(
                    f"          peak circuit MLP MI  = {peak_mlp['mlp_mi']:.3f} "
                    f"(L{peak_mlp['layer']})"
                )

df_circuit_component_mi = pd.DataFrame(component_mi_records)
if len(df_circuit_component_mi) > 0:
    save_analysis_circuit(df_circuit_component_mi, "06c_circuit_component_mi.csv")
    print(f"\nCircuit component MI records: {len(df_circuit_component_mi)}")
else:
    print("\nNo circuit component MI data available.")


pythia-70m:


  draw_1: peak circuit attn MI = 1.002 (L1)
          peak circuit MLP MI  = 0.615 (L0)



pythia-160m:


  draw_1: peak circuit attn MI = 1.026 (L6)
          peak circuit MLP MI  = 0.757 (L0)



pythia-410m:


...


pythia-1b:


  draw_1: peak circuit attn MI = 1.515 (L15)
          peak circuit MLP MI  = 0.992 (L12)



pythia-1.4b:


  draw_1: peak circuit attn MI = 1.718 (L21)
          peak circuit MLP MI  = 1.034 (L22)



Circuit component MI records: 246


In [14]:
# Plot circuit component MI per model
if len(df_circuit_component_mi) > 0:
    for model in MODELS:
        model_data = df_circuit_component_mi[
            (df_circuit_component_mi["model"] == model)
            & (df_circuit_component_mi["draw"] == "draw_1")
        ].sort_values("layer")
        if len(model_data) == 0:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Attn vs MLP MI trajectories
        axes[0].plot(
            model_data["layer"],
            model_data["attn_mi"],
            color="#1f77b4",
            label="Attention",
            marker="o",
            markersize=4,
        )
        axes[0].plot(
            model_data["layer"],
            model_data["mlp_mi"],
            color="#ff7f0e",
            label="MLP",
            marker="s",
            markersize=4,
        )
        axes[0].set_xlabel("Layer")
        axes[0].set_ylabel("MI (bits)")
        axes[0].set_title(f"Circuit Component MI -- {model}")
        axes[0].legend()
        axes[0].set_ylim(bottom=0)

        # Right: MLP advantage (MLP MI - Attn MI)
        colors = [
            "#ff7f0e" if v >= 0 else "#1f77b4" for v in model_data["mlp_advantage"]
        ]
        axes[1].bar(
            model_data["layer"], model_data["mlp_advantage"], color=colors, alpha=0.8
        )
        axes[1].axhline(y=0, color="black", linewidth=0.5)
        axes[1].set_xlabel("Layer")
        axes[1].set_ylabel("MLP MI - Attn MI (bits)")
        axes[1].set_title(f"Circuit MLP Information Advantage -- {model}")

        fig.tight_layout()
        save_figure_circuit(fig, f"viz_06c_05_circuit_component_mi_{model}.png")
else:
    print("No circuit component MI data -- skipping visualization.")

## 6. Circuit Coding Efficiency

Bits of band information per dimension under circuit mode:
efficiency = MI / d_model.
Normalizes MI by representational capacity to compare across models.

In [15]:
efficiency_records = []

for model in MODELS:
    d_model = MODEL_D_MODEL[model]

    for draw in DRAWS:
        model_draw_mi = df_circuit_mi_combined[
            (df_circuit_mi_combined["model"] == model)
            & (df_circuit_mi_combined["draw"] == draw)
        ].sort_values("layer")

        if len(model_draw_mi) == 0:
            continue

        for _, row in model_draw_mi.iterrows():
            mi_val = row["mi_ksg"]
            if np.isnan(mi_val):
                continue

            eff = compute_coding_efficiency(mi_val, d_model)

            efficiency_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": row["layer"],
                    "mi": mi_val,
                    "efficiency": eff,
                    "d_model": d_model,
                    "model_capacity": MODEL_CAPACITY[model],
                }
            )

df_circuit_efficiency = pd.DataFrame(efficiency_records)
save_analysis_circuit(df_circuit_efficiency, "06c_circuit_coding_efficiency.csv")
print(f"Circuit coding efficiency records: {len(df_circuit_efficiency)}")

# Summary: peak efficiency per model
for model in MODELS:
    model_data = df_circuit_efficiency[
        (df_circuit_efficiency["model"] == model)
        & (df_circuit_efficiency["draw"] == "draw_1")
    ]
    if len(model_data) > 0:
        peak = model_data.loc[model_data["efficiency"].idxmax()]
        print(
            f"{model} (d={MODEL_D_MODEL[model]}): peak circuit eff = "
            f"{peak['efficiency']:.6f} bits/dim at layer {int(peak['layer'])} "
            f"(MI = {peak['mi']:.3f} bits)"
        )

Circuit coding efficiency records: 246
pythia-70m (d=512): peak circuit eff = 0.001482 bits/dim at layer 0 (MI = 0.759 bits)
pythia-160m (d=768): peak circuit eff = 0.001171 bits/dim at layer 6 (MI = 0.899 bits)
pythia-410m (d=1024): peak circuit eff = 0.000955 bits/dim at layer 23 (MI = 0.978 bits)
pythia-1b (d=2048): peak circuit eff = 0.000598 bits/dim at layer 13 (MI = 1.225 bits)
pythia-1.4b (d=2048): peak circuit eff = 0.000586 bits/dim at layer 22 (MI = 1.200 bits)


In [16]:
# Plot circuit coding efficiency trajectories
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Efficiency trajectories (absolute layer)
for model in MODELS:
    model_data = df_circuit_efficiency[
        (df_circuit_efficiency["model"] == model)
        & (df_circuit_efficiency["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    axes[0].plot(
        model_data["layer"],
        model_data["efficiency"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Coding Efficiency (bits/dim)")
axes[0].set_title("Circuit Coding Efficiency Trajectory")
axes[0].legend()
axes[0].set_ylim(bottom=0)

# Right: Peak efficiency vs model capacity
peak_eff_data = []
for model in MODELS:
    for draw in DRAWS:
        md = df_circuit_efficiency[
            (df_circuit_efficiency["model"] == model)
            & (df_circuit_efficiency["draw"] == draw)
        ]
        if len(md) > 0:
            peak_eff_data.append(
                {
                    "model": model,
                    "draw": draw,
                    "model_capacity": MODEL_CAPACITY[model],
                    "peak_efficiency": md["efficiency"].max(),
                }
            )

df_peak_eff = pd.DataFrame(peak_eff_data)
for model in MODELS:
    md = df_peak_eff[df_peak_eff["model"] == model]
    if len(md) > 0:
        axes[1].scatter(
            md["model_capacity"],
            md["peak_efficiency"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=60,
            zorder=5,
        )

# Trend line (mean per model)
if len(df_peak_eff) > 0:
    means = (
        df_peak_eff.groupby("model")
        .agg({"model_capacity": "first", "peak_efficiency": "mean"})
        .sort_values("model_capacity")
    )
    if len(means) > 1:
        axes[1].plot(
            means["model_capacity"], means["peak_efficiency"], "k--", alpha=0.3
        )

axes[1].set_xlabel("Model Capacity (M params)")
axes[1].set_ylabel("Peak Coding Efficiency (bits/dim)")
axes[1].set_title("Circuit Peak Efficiency vs Model Size")
axes[1].set_xscale("log")
axes[1].legend()

fig.tight_layout()
save_figure_circuit(fig, "viz_06c_06_circuit_coding_efficiency.png")

## 7. Base vs Circuit Comparison

Overlay base and circuit MI trajectories, compute information loss per layer,
and compare coding efficiency between full model and circuit.

In [17]:
# Load base info-theoretic results
try:
    df_base_mi_ksg = load_domain_csv(
        "info_theoretic", "base", "06_mi_ksg_trajectory.csv"
    )
    print(f"Loaded base MI (KSG) trajectory: {len(df_base_mi_ksg)} rows")
except FileNotFoundError:
    df_base_mi_ksg = pd.DataFrame()
    print("Base MI (KSG) trajectory not found.")

try:
    df_base_mi_probe = load_domain_csv(
        "info_theoretic", "base", "06_mi_probe_trajectory.csv"
    )
    print(f"Loaded base MI (probe) trajectory: {len(df_base_mi_probe)} rows")
except FileNotFoundError:
    df_base_mi_probe = pd.DataFrame()
    print("Base MI (probe) trajectory not found.")

try:
    df_base_efficiency = load_domain_csv(
        "info_theoretic", "base", "06_coding_efficiency.csv"
    )
    print(f"Loaded base coding efficiency: {len(df_base_efficiency)} rows")
except FileNotFoundError:
    df_base_efficiency = pd.DataFrame()
    print("Base coding efficiency not found.")

Loaded base MI (KSG) trajectory: 174 rows
Loaded base MI (probe) trajectory: 174 rows
Loaded base coding efficiency: 174 rows


In [18]:
# MI trajectory overlay: base vs circuit (KSG)
if not df_base_mi_ksg.empty and len(df_circuit_mi_ksg) > 0:
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(12, 6))

        # Base (solid)
        df_base_m = df_base_mi_ksg[
            (df_base_mi_ksg["model"] == model) & (df_base_mi_ksg["draw"] == "draw_1")
        ].sort_values("layer")
        if not df_base_m.empty:
            ax.plot(
                df_base_m["layer"],
                df_base_m["mi_ksg"],
                color="steelblue",
                label="Base",
                linewidth=2,
                marker="o",
                markersize=4,
            )

        # Circuit (dashed)
        df_circ_m = df_circuit_mi_combined[
            (df_circuit_mi_combined["model"] == model)
            & (df_circuit_mi_combined["draw"] == "draw_1")
        ].sort_values("layer")
        if not df_circ_m.empty:
            ax.plot(
                df_circ_m["layer"],
                df_circ_m["mi_ksg"],
                color="coral",
                label="Circuit",
                linewidth=2,
                linestyle="--",
                marker="s",
                markersize=4,
            )

        ax.axhline(
            y=H_Y, color="gray", linestyle=":", alpha=0.5, label=f"H(Y)={H_Y:.2f}"
        )
        ax.set_xlabel("Layer")
        ax.set_ylabel("MI (KSG, bits)")
        ax.set_title(f"Base vs Circuit MI Trajectory -- {model}")
        ax.legend()
        ax.set_ylim(bottom=0)
        fig.tight_layout()
        save_figure_comp(fig, f"viz_06c_07_mi_overlay_{model}.png")
else:
    print("Insufficient data for base vs circuit MI overlay.")

In [19]:
# Information loss per layer: base_MI - circuit_MI
info_loss_records = []

if not df_base_mi_ksg.empty and len(df_circuit_mi_combined) > 0:
    for model in MODELS:
        for draw in DRAWS:
            base_data = df_base_mi_ksg[
                (df_base_mi_ksg["model"] == model) & (df_base_mi_ksg["draw"] == draw)
            ][["layer", "mi_ksg"]].rename(columns={"mi_ksg": "base_mi"})

            circuit_data = df_circuit_mi_combined[
                (df_circuit_mi_combined["model"] == model)
                & (df_circuit_mi_combined["draw"] == draw)
            ][["layer", "mi_ksg"]].rename(columns={"mi_ksg": "circuit_mi"})

            if base_data.empty or circuit_data.empty:
                continue

            merged = base_data.merge(circuit_data, on="layer")
            if merged.empty:
                continue

            for _, row in merged.iterrows():
                loss = row["base_mi"] - row["circuit_mi"]
                frac_loss = loss / row["base_mi"] if row["base_mi"] > 0 else 0.0
                info_loss_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "layer": row["layer"],
                        "base_mi": row["base_mi"],
                        "circuit_mi": row["circuit_mi"],
                        "info_loss": loss,
                        "fractional_loss": frac_loss,
                    }
                )

df_info_loss = pd.DataFrame(info_loss_records)
if len(df_info_loss) > 0:
    save_analysis_comp(df_info_loss, "06c_information_loss.csv")
    print(f"Information loss records: {len(df_info_loss)}")

    # Summary per model
    for model in MODELS:
        md = df_info_loss[
            (df_info_loss["model"] == model) & (df_info_loss["draw"] == "draw_1")
        ]
        if len(md) > 0:
            max_loss = md.loc[md["info_loss"].idxmax()]
            mean_frac = md["fractional_loss"].mean()
            print(
                f"{model}: max info loss = {max_loss['info_loss']:.3f} bits "
                f"at layer {int(max_loss['layer'])}, "
                f"mean fractional loss = {mean_frac:.3f}"
            )
else:
    print("No information loss data computed.")

Information loss records: 174
pythia-70m: max info loss = 0.015 bits at layer 1, mean fractional loss = 0.007
pythia-160m: max info loss = 0.044 bits at layer 3, mean fractional loss = 0.020
pythia-410m: max info loss = 0.021 bits at layer 5, mean fractional loss = -0.010
pythia-1b: max info loss = 0.049 bits at layer 6, mean fractional loss = -0.002


In [20]:
# Visualize information loss per model
if len(df_info_loss) > 0:
    for model in MODELS:
        model_data = df_info_loss[
            (df_info_loss["model"] == model) & (df_info_loss["draw"] == "draw_1")
        ].sort_values("layer")
        if len(model_data) == 0:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Left: absolute information loss
        colors = ["#d62728" if v > 0 else "#2ca02c" for v in model_data["info_loss"]]
        axes[0].bar(
            model_data["layer"], model_data["info_loss"], color=colors, alpha=0.8
        )
        axes[0].axhline(y=0, color="black", linewidth=0.5)
        axes[0].set_xlabel("Layer")
        axes[0].set_ylabel("Info Loss (bits): base_MI - circuit_MI")
        axes[0].set_title(f"Information Loss per Layer -- {model}")

        # Right: fractional loss
        axes[1].plot(
            model_data["layer"],
            model_data["fractional_loss"],
            color="#d62728",
            marker="o",
            markersize=4,
        )
        axes[1].axhline(y=0, color="black", linewidth=0.5)
        axes[1].set_xlabel("Layer")
        axes[1].set_ylabel("Fractional Loss (base_MI - circuit_MI) / base_MI")
        axes[1].set_title(f"Fractional Information Loss -- {model}")

        fig.tight_layout()
        save_figure_comp(fig, f"viz_06c_08_info_loss_{model}.png")

    # Cross-model information loss (normalized depth)
    fig, ax = plt.subplots(figsize=(14, 7))
    for model in MODELS:
        model_data = df_info_loss[
            (df_info_loss["model"] == model) & (df_info_loss["draw"] == "draw_1")
        ].sort_values("layer")
        if len(model_data) == 0:
            continue
        n_layers = MODEL_INFO[model]["n_layers"]
        layer_frac = model_data["layer"] / max(n_layers - 1, 1)
        ax.plot(
            layer_frac,
            model_data["info_loss"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            marker="o",
            markersize=4,
        )

    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_xlabel("Relative Layer Position")
    ax.set_ylabel("Info Loss (bits)")
    ax.set_title("Information Loss Across Models (Normalized Depth)")
    ax.legend()
    fig.tight_layout()
    save_figure_comp(fig, "viz_06c_09_info_loss_all_models.png")
else:
    print("No information loss data -- skipping visualization.")

In [21]:
# Coding efficiency comparison: base vs circuit
eff_comp_records = []

if not df_base_efficiency.empty and len(df_circuit_efficiency) > 0:
    for model in MODELS:
        for draw in DRAWS:
            base_eff = df_base_efficiency[
                (df_base_efficiency["model"] == model)
                & (df_base_efficiency["draw"] == draw)
            ][["layer", "efficiency"]].rename(columns={"efficiency": "base_efficiency"})

            circ_eff = df_circuit_efficiency[
                (df_circuit_efficiency["model"] == model)
                & (df_circuit_efficiency["draw"] == draw)
            ][["layer", "efficiency"]].rename(
                columns={"efficiency": "circuit_efficiency"}
            )

            if base_eff.empty or circ_eff.empty:
                continue

            merged = base_eff.merge(circ_eff, on="layer")
            for _, row in merged.iterrows():
                delta_eff = row["circuit_efficiency"] - row["base_efficiency"]
                eff_comp_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "layer": row["layer"],
                        "base_efficiency": row["base_efficiency"],
                        "circuit_efficiency": row["circuit_efficiency"],
                        "delta_efficiency": delta_eff,
                    }
                )

df_eff_comp = pd.DataFrame(eff_comp_records)
if len(df_eff_comp) > 0:
    save_analysis_comp(df_eff_comp, "06c_efficiency_comparison.csv")
    print(f"Efficiency comparison records: {len(df_eff_comp)}")

    # Summary
    for model in MODELS:
        md = df_eff_comp[
            (df_eff_comp["model"] == model) & (df_eff_comp["draw"] == "draw_1")
        ]
        if len(md) > 0:
            mean_delta = md["delta_efficiency"].mean()
            print(
                f"{model}: mean delta efficiency = {mean_delta:.6f} bits/dim "
                f"({'decrease' if mean_delta < 0 else 'increase'})"
            )
else:
    print("No efficiency comparison data.")

Efficiency comparison records: 174
pythia-70m: mean delta efficiency = -0.000009 bits/dim (decrease)
pythia-160m: mean delta efficiency = -0.000021 bits/dim (decrease)
pythia-410m: mean delta efficiency = 0.000008 bits/dim (increase)
pythia-1b: mean delta efficiency = 0.000002 bits/dim (increase)


In [22]:
# Visualize efficiency comparison: base vs circuit overlay
if len(df_eff_comp) > 0:
    for model in MODELS:
        model_data = df_eff_comp[
            (df_eff_comp["model"] == model) & (df_eff_comp["draw"] == "draw_1")
        ].sort_values("layer")
        if len(model_data) == 0:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Overlay
        axes[0].plot(
            model_data["layer"],
            model_data["base_efficiency"],
            color="steelblue",
            label="Base",
            linewidth=2,
            marker="o",
            markersize=4,
        )
        axes[0].plot(
            model_data["layer"],
            model_data["circuit_efficiency"],
            color="coral",
            label="Circuit",
            linewidth=2,
            linestyle="--",
            marker="s",
            markersize=4,
        )
        axes[0].set_xlabel("Layer")
        axes[0].set_ylabel("Coding Efficiency (bits/dim)")
        axes[0].set_title(f"Coding Efficiency: Base vs Circuit -- {model}")
        axes[0].legend()
        axes[0].set_ylim(bottom=0)

        # Right: Delta efficiency
        colors = [
            "#d62728" if v < 0 else "#2ca02c" for v in model_data["delta_efficiency"]
        ]
        axes[1].bar(
            model_data["layer"], model_data["delta_efficiency"], color=colors, alpha=0.8
        )
        axes[1].axhline(y=0, color="black", linewidth=0.5)
        axes[1].set_xlabel("Layer")
        axes[1].set_ylabel("Delta Efficiency (circuit - base, bits/dim)")
        axes[1].set_title(f"Efficiency Change Under Ablation -- {model}")

        fig.tight_layout()
        save_figure_comp(fig, f"viz_06c_10_efficiency_comparison_{model}.png")
else:
    print("No efficiency comparison data -- skipping visualization.")

## 8. Integration: MI-based vs Geometric Circuit Impact

Do MI-based and geometric measures agree on circuit impact?
Correlate information loss from this notebook with CKA degradation
from NB02c (residual stream circuit analysis).

In [23]:
# Load CKA from NB02c (requires NB02c to have been run first)
cka_path = Path("outputs/residual_stream/comparison/analysis/02c_base_circuit_cka.csv")
try:
    df_cka = load_domain_csv(
        "residual_stream", "comparison", "02c_base_circuit_cka.csv"
    )
    print(f"Loaded base-circuit CKA: {len(df_cka)} rows")
    print(f"  Columns: {list(df_cka.columns)}")
except FileNotFoundError:
    df_cka = pd.DataFrame()
    print(f"CKA data not found at {cka_path}")
    print(
        "  -> Run NB02c (02c_residual_circuit.ipynb) first, then re-run this notebook."
    )

Loaded base-circuit CKA: 870 rows
  Columns: ['model', 'band', 'draw', 'layer', 'cka_base_circuit', 'mean_delta_norm']


In [24]:
from scipy import stats as sp_stats

integration_records = []

if not df_cka.empty and len(df_info_loss) > 0:
    # Average CKA across bands per model/draw/layer
    df_cka_agg = (
        df_cka.groupby(["model", "draw", "layer"])
        .agg(
            mean_cka=("cka_base_circuit", "mean"),
            mean_delta_norm=("mean_delta_norm", "mean"),
        )
        .reset_index()
    )

    for model in MODELS:
        for draw in DRAWS:
            # Information loss
            loss_data = df_info_loss[
                (df_info_loss["model"] == model) & (df_info_loss["draw"] == draw)
            ][["layer", "info_loss", "fractional_loss"]]

            # CKA
            cka_data = df_cka_agg[
                (df_cka_agg["model"] == model) & (df_cka_agg["draw"] == draw)
            ][["layer", "mean_cka", "mean_delta_norm"]]

            if loss_data.empty or cka_data.empty:
                continue

            merged = loss_data.merge(cka_data, on="layer")
            if len(merged) < 3:
                continue

            # Correlate info_loss with 1-CKA (CKA degradation)
            merged["cka_degradation"] = 1.0 - merged["mean_cka"]

            r_cka, p_cka = sp_stats.pearsonr(
                merged["info_loss"], merged["cka_degradation"]
            )
            rho_cka, p_rho_cka = sp_stats.spearmanr(
                merged["info_loss"], merged["cka_degradation"]
            )

            # Correlate info_loss with delta_norm
            r_norm, p_norm = sp_stats.pearsonr(
                merged["info_loss"], merged["mean_delta_norm"]
            )
            rho_norm, p_rho_norm = sp_stats.spearmanr(
                merged["info_loss"], merged["mean_delta_norm"]
            )

            integration_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "comparison": "info_loss_vs_cka_degradation",
                    "pearson_r": r_cka,
                    "pearson_p": p_cka,
                    "spearman_rho": rho_cka,
                    "spearman_p": p_rho_cka,
                    "n_layers": len(merged),
                }
            )
            integration_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "comparison": "info_loss_vs_delta_norm",
                    "pearson_r": r_norm,
                    "pearson_p": p_norm,
                    "spearman_rho": rho_norm,
                    "spearman_p": p_rho_norm,
                    "n_layers": len(merged),
                }
            )

    print("Information Loss vs CKA Degradation:")
    for rec in integration_records:
        if rec["comparison"] == "info_loss_vs_cka_degradation":
            sig = "*" if rec["pearson_p"] < 0.05 else ""
            print(
                f"  {rec['model']}/{rec['draw']}: r={rec['pearson_r']:.3f}{sig}, "
                f"rho={rec['spearman_rho']:.3f}"
            )

    print("\nInformation Loss vs Delta Norm:")
    for rec in integration_records:
        if rec["comparison"] == "info_loss_vs_delta_norm":
            sig = "*" if rec["pearson_p"] < 0.05 else ""
            print(
                f"  {rec['model']}/{rec['draw']}: r={rec['pearson_r']:.3f}{sig}, "
                f"rho={rec['spearman_rho']:.3f}"
            )
else:
    print("Insufficient data for MI-CKA integration.")

# Save integration results
if integration_records:
    df_integration = pd.DataFrame(integration_records)
    save_analysis_comp(df_integration, "06c_mi_geometric_integration.csv")
    print(f"\nIntegration records: {len(df_integration)}")
    print("\nSummary by comparison type:")
    print(
        df_integration.groupby("comparison")[["pearson_r", "spearman_rho"]]
        .mean()
        .round(3)
    )

Information Loss vs CKA Degradation:
  pythia-70m/draw_1: r=0.196, rho=0.257
  pythia-70m/draw_2: r=0.206, rho=-0.086
  pythia-70m/draw_3: r=-0.084, rho=0.029
  pythia-160m/draw_1: r=-0.199, rho=-0.280
  pythia-160m/draw_2: r=-0.159, rho=-0.161
  pythia-160m/draw_3: r=0.102, rho=0.049
  pythia-410m/draw_1: r=-0.724*, rho=-0.792
  pythia-410m/draw_2: r=-0.635*, rho=-0.765
  pythia-410m/draw_3: r=-0.152, rho=-0.450
  pythia-1b/draw_1: r=-0.368, rho=-0.271
  pythia-1b/draw_2: r=-0.392, rho=-0.503
  pythia-1b/draw_3: r=-0.582*, rho=-0.809

Information Loss vs Delta Norm:
  pythia-70m/draw_1: r=0.160, rho=0.143
  pythia-70m/draw_2: r=0.203, rho=-0.086
  pythia-70m/draw_3: r=0.046, rho=-0.086
  pythia-160m/draw_1: r=-0.075, rho=-0.147
  pythia-160m/draw_2: r=-0.051, rho=-0.189
  pythia-160m/draw_3: r=0.223, rho=0.098
  pythia-410m/draw_1: r=-0.751*, rho=-0.819
  pythia-410m/draw_2: r=-0.693*, rho=-0.703
  pythia-410m/draw_3: r=-0.261, rho=-0.390
  pythia-1b/draw_1: r=-0.849*, rho=-0.853
  py

In [25]:
# Visualize MI info loss vs CKA degradation scatter (per model)
if not df_cka.empty and len(df_info_loss) > 0:
    df_cka_agg = (
        df_cka.groupby(["model", "draw", "layer"])
        .agg(
            mean_cka=("cka_base_circuit", "mean"),
        )
        .reset_index()
    )

    fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True)
    if len(MODELS) == 1:
        axes = [axes]

    for ax, model in zip(axes, MODELS):
        loss_data = df_info_loss[
            (df_info_loss["model"] == model) & (df_info_loss["draw"] == "draw_1")
        ][["layer", "info_loss"]]

        cka_data = df_cka_agg[
            (df_cka_agg["model"] == model) & (df_cka_agg["draw"] == "draw_1")
        ][["layer", "mean_cka"]]

        merged = loss_data.merge(cka_data, on="layer")
        if len(merged) < 2:
            ax.set_title(f"{model} (insufficient data)")
            continue

        merged["cka_degradation"] = 1.0 - merged["mean_cka"]

        scatter = ax.scatter(
            merged["info_loss"],
            merged["cka_degradation"],
            c=merged["layer"],
            cmap="viridis",
            s=50,
            edgecolors="black",
            linewidth=0.5,
        )
        plt.colorbar(scatter, ax=ax, label="Layer")

        if len(merged) > 2:
            r, _ = sp_stats.pearsonr(merged["info_loss"], merged["cka_degradation"])
            ax.set_title(f"{model} (r={r:.3f})")
        else:
            ax.set_title(model)

        ax.set_xlabel("Info Loss (bits)")
        if ax == axes[0]:
            ax.set_ylabel("CKA Degradation (1 - CKA)")

    fig.suptitle("MI Information Loss vs CKA Degradation (colored by layer)", y=1.02)
    fig.tight_layout()
    save_figure_comp(fig, "viz_06c_11_mi_vs_cka_scatter.png")
else:
    print("Insufficient data for MI-CKA scatter plot.")

In [26]:
# Dual-axis overlay: info loss and CKA degradation (per model)
if not df_cka.empty and len(df_info_loss) > 0:
    df_cka_agg = (
        df_cka.groupby(["model", "draw", "layer"])
        .agg(
            mean_cka=("cka_base_circuit", "mean"),
        )
        .reset_index()
    )

    for model in MODELS:
        loss_data = df_info_loss[
            (df_info_loss["model"] == model) & (df_info_loss["draw"] == "draw_1")
        ].sort_values("layer")
        cka_data = df_cka_agg[
            (df_cka_agg["model"] == model) & (df_cka_agg["draw"] == "draw_1")
        ].sort_values("layer")

        if loss_data.empty or cka_data.empty:
            continue

        fig, ax1 = plt.subplots(figsize=(12, 6))
        color1 = "#d62728"
        ax1.plot(
            loss_data["layer"],
            loss_data["info_loss"],
            color=color1,
            marker="o",
            markersize=4,
            label="Info Loss (bits)",
        )
        ax1.set_xlabel("Layer")
        ax1.set_ylabel("Info Loss (bits)", color=color1)
        ax1.tick_params(axis="y", labelcolor=color1)

        ax2 = ax1.twinx()
        color2 = "#1f77b4"
        cka_data_merged = cka_data.copy()
        cka_data_merged["cka_deg"] = 1.0 - cka_data_merged["mean_cka"]
        ax2.plot(
            cka_data_merged["layer"],
            cka_data_merged["cka_deg"],
            color=color2,
            marker="s",
            markersize=4,
            label="CKA Degradation (1-CKA)",
        )
        ax2.set_ylabel("CKA Degradation", color=color2)
        ax2.tick_params(axis="y", labelcolor=color2)

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

        fig.suptitle(f"Info Loss vs CKA Degradation -- {model}")
        fig.tight_layout()
        save_figure_comp(fig, f"viz_06c_12_info_loss_vs_cka_{model}.png")
else:
    print("Insufficient data for dual-axis overlay.")

## 9. Summary

In [27]:
# Build master circuit info-theoretic summary
master_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_model = MODEL_D_MODEL[model]

    for draw in DRAWS:
        record = {
            "model": model,
            "draw": draw,
            "model_capacity": MODEL_CAPACITY[model],
            "n_layers": n_layers,
            "d_model": d_model,
        }

        # Peak circuit KSG MI
        mi_data = df_circuit_mi_combined[
            (df_circuit_mi_combined["model"] == model)
            & (df_circuit_mi_combined["draw"] == draw)
        ]
        if len(mi_data) > 0 and mi_data["mi_ksg"].notna().any():
            peak_idx = mi_data["mi_ksg"].idxmax()
            record["peak_mi_ksg"] = mi_data.loc[peak_idx, "mi_ksg"]
            record["peak_mi_ksg_layer"] = mi_data.loc[peak_idx, "layer"]
            record["peak_mi_ksg_layer_frac"] = mi_data.loc[peak_idx, "layer"] / max(
                n_layers - 1, 1
            )

        # Peak circuit delta-MI
        delta_data = df_circuit_delta_mi[
            (df_circuit_delta_mi["model"] == model)
            & (df_circuit_delta_mi["draw"] == draw)
        ]
        if len(delta_data) > 0:
            peak_idx = delta_data["delta_mi"].idxmax()
            record["peak_delta_mi"] = delta_data.loc[peak_idx, "delta_mi"]
            record["peak_delta_mi_layer"] = delta_data.loc[peak_idx, "layer"]

        # Peak circuit coding efficiency
        eff_data = df_circuit_efficiency[
            (df_circuit_efficiency["model"] == model)
            & (df_circuit_efficiency["draw"] == draw)
        ]
        if len(eff_data) > 0:
            record["peak_efficiency"] = eff_data["efficiency"].max()

        # Min circuit conditional entropy
        cond_data = df_circuit_cond_ent[
            (df_circuit_cond_ent["model"] == model)
            & (df_circuit_cond_ent["draw"] == draw)
        ]
        if len(cond_data) > 0:
            record["min_h_y_given_x"] = cond_data["h_y_given_x"].min()
            record["max_info_fraction"] = cond_data["info_fraction"].max()

        # Mean info loss (if comparison available)
        if len(df_info_loss) > 0:
            loss_data = df_info_loss[
                (df_info_loss["model"] == model) & (df_info_loss["draw"] == draw)
            ]
            if len(loss_data) > 0:
                record["mean_info_loss"] = loss_data["info_loss"].mean()
                record["max_info_loss"] = loss_data["info_loss"].max()
                record["mean_fractional_loss"] = loss_data["fractional_loss"].mean()

        master_records.append(record)

df_master = pd.DataFrame(master_records)
save_analysis_circuit(df_master, "06c_master_circuit_info_theoretic.csv")

# Display summary
summary_cols = [
    "model",
    "peak_mi_ksg",
    "peak_mi_ksg_layer_frac",
    "peak_delta_mi",
    "peak_efficiency",
    "max_info_fraction",
    "mean_info_loss",
]
existing_cols = [c for c in summary_cols if c in df_master.columns]
print("Cross-model circuit info-theoretic summary (mean over draws):")
print(df_master.groupby("model")[existing_cols[1:]].mean().round(4))

Cross-model circuit info-theoretic summary (mean over draws):
             peak_mi_ksg  peak_mi_ksg_layer_frac  peak_delta_mi  \
model                                                             
pythia-1.4b       1.1532                  0.3623         0.9877   
pythia-160m       0.9058                  0.5455         0.8598   
pythia-1b         1.2043                  0.8667         1.1164   
pythia-410m       0.9899                  0.7101         0.8687   
pythia-70m        0.7251                  0.0000         0.7251   

             peak_efficiency  max_info_fraction  mean_info_loss  
model                                                            
pythia-1.4b           0.0006             0.4967             NaN  
pythia-160m           0.0012             0.3901          0.0155  
pythia-1b             0.0006             0.5186         -0.0104  
pythia-410m           0.0010             0.4263         -0.0105  
pythia-70m            0.0014             0.3123          0.0041  


In [28]:
# Scaling panel: circuit peak MI, peak delta-MI, peak efficiency vs model size
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scaling_metrics = [
    ("peak_mi_ksg", "Circuit Peak MI (KSG, bits)"),
    ("peak_delta_mi", "Circuit Peak Delta-MI (bits)"),
    ("peak_efficiency", "Circuit Peak Efficiency (bits/dim)"),
]

for ax, (metric, title) in zip(axes, scaling_metrics):
    if metric not in df_master.columns:
        ax.set_title(f"{title} (N/A)")
        continue
    for model in MODELS:
        md = df_master[df_master["model"] == model]
        if len(md) == 0 or metric not in md.columns:
            continue
        ax.scatter(
            md["model_capacity"],
            md[metric],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=60,
            zorder=5,
        )
    # Mean trend line
    means = (
        df_master.groupby("model")
        .agg({"model_capacity": "first", metric: "mean"})
        .dropna()
        .sort_values("model_capacity")
    )
    if len(means) > 1:
        ax.plot(means["model_capacity"], means[metric], "k--", alpha=0.3)
    ax.set_xlabel("Model Capacity (M params)")
    ax.set_xscale("log")
    ax.set_title(title)

axes[0].legend(loc="best", fontsize=8)
fig.tight_layout()
save_figure_circuit(fig, "viz_06c_13_circuit_scaling_panel.png")

In [29]:
# Draw stability analysis
stability_records = []
for model in MODELS:
    model_data = df_master[df_master["model"] == model]
    if len(model_data) < 2:
        continue

    stability_metrics = [
        "peak_mi_ksg",
        "peak_mi_ksg_layer_frac",
        "peak_delta_mi",
        "peak_efficiency",
        "max_info_fraction",
    ]

    for metric in stability_metrics:
        if metric not in model_data.columns:
            continue
        vals = model_data[metric].dropna()
        if len(vals) < 2:
            continue
        stability_records.append(
            {
                "model": model,
                "metric": metric,
                "mean": float(vals.mean()),
                "std": float(vals.std()),
                "cv": float(vals.std() / vals.mean()) if vals.mean() != 0 else np.nan,
            }
        )

df_stability = pd.DataFrame(stability_records)
if len(df_stability) > 0:
    save_analysis_circuit(df_stability, "06c_circuit_draw_stability.csv")
    print("Circuit draw stability (CV = coefficient of variation):")
    pivot = df_stability.pivot(index="model", columns="metric", values="cv")
    print(pivot.round(4))
else:
    print("Insufficient data for stability analysis.")

Circuit draw stability (CV = coefficient of variation):
metric       max_info_fraction  peak_delta_mi  peak_efficiency  peak_mi_ksg  \
model                                                                         
pythia-1.4b             0.0403         0.0142           0.0403       0.0403   
pythia-160m             0.0152         0.0232           0.0152       0.0152   
pythia-1b               0.0150         0.0307           0.0150       0.0150   
pythia-410m             0.0284         0.0252           0.0284       0.0284   
pythia-70m              0.0490         0.0490           0.0490       0.0490   

metric       peak_mi_ksg_layer_frac  
model                                
pythia-1.4b                  1.4215  
pythia-160m                  0.0000  
pythia-1b                    0.0000  
pythia-410m                  0.7070  
pythia-70m                      NaN  


In [30]:
print("\n" + "=" * 70)
print("NOTEBOOK 06c COMPLETE: Information-Theoretic Circuit Analysis")
print("=" * 70)

print(f"\nCircuit CSVs in: {CIRCUIT_ANALYSIS}")
for f in sorted(CIRCUIT_ANALYSIS.glob("06c_*")):
    print(f"  {f.name}")

print(f"\nComparison CSVs in: {COMP_ANALYSIS}")
for f in sorted(COMP_ANALYSIS.glob("06c_*")):
    print(f"  {f.name}")

print(f"\nCircuit figures in: {CIRCUIT_VIZ}")
for f in sorted(CIRCUIT_VIZ.glob("viz_06c_*")):
    print(f"  {f.name}")

print(f"\nComparison figures in: {COMP_VIZ}")
for f in sorted(COMP_VIZ.glob("viz_06c_*")):
    print(f"  {f.name}")

print("\n--- Key Findings ---")
for model in MODELS:
    md = df_master[(df_master["model"] == model) & (df_master["draw"] == "draw_1")]
    if len(md) == 0:
        continue
    row = md.iloc[0]
    lines = [f"\n{model} (d_model={MODEL_D_MODEL[model]}):"]
    if "peak_mi_ksg" in row and not np.isnan(row.get("peak_mi_ksg", np.nan)):
        lines.append(
            f"  Circuit peak MI (KSG) = {row['peak_mi_ksg']:.3f} bits "
            f"at {row['peak_mi_ksg_layer_frac']:.0%} depth"
        )
    if "peak_delta_mi" in row and not np.isnan(row.get("peak_delta_mi", np.nan)):
        lines.append(f"  Circuit peak delta-MI = {row['peak_delta_mi']:.4f} bits")
    if "peak_efficiency" in row and not np.isnan(row.get("peak_efficiency", np.nan)):
        lines.append(
            f"  Circuit peak efficiency = {row['peak_efficiency']:.6f} bits/dim"
        )
    if "max_info_fraction" in row and not np.isnan(
        row.get("max_info_fraction", np.nan)
    ):
        lines.append(f"  Circuit max info fraction = {row['max_info_fraction']:.3f}")
    if "mean_info_loss" in row and not np.isnan(row.get("mean_info_loss", np.nan)):
        lines.append(f"  Mean info loss vs base = {row['mean_info_loss']:.3f} bits")
    if "mean_fractional_loss" in row and not np.isnan(
        row.get("mean_fractional_loss", np.nan)
    ):
        lines.append(f"  Mean fractional loss   = {row['mean_fractional_loss']:.3f}")
    print("\n".join(lines))

# Report integration correlations
if integration_records:
    print("\n--- MI-Geometric Integration ---")
    df_int = pd.DataFrame(integration_records)
    for comparison in df_int["comparison"].unique():
        cd = df_int[df_int["comparison"] == comparison]
        mean_r = cd["pearson_r"].mean()
        mean_rho = cd["spearman_rho"].mean()
        print(f"  {comparison}: mean r={mean_r:.3f}, mean rho={mean_rho:.3f}")


NOTEBOOK 06c COMPLETE: Information-Theoretic Circuit Analysis

Circuit CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/circuit/analysis
  06c_circuit_coding_efficiency.csv
  06c_circuit_component_mi.csv
  06c_circuit_conditional_entropy.csv
  06c_circuit_delta_mi.csv
  06c_circuit_draw_stability.csv
  06c_circuit_mi_trajectory.csv
  06c_master_circuit_info_theoretic.csv

Comparison CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/comparison/analysis
  06c_efficiency_comparison.csv
  06c_information_loss.csv
  06c_mi_geometric_integration.csv

Circuit figures in: LSC_circuit_analysis/03_Phase_Representational/outputs/info_theoretic/circuit/viz
  viz_06c_01_circuit_mi_trajectory_pythia-1.4b.png
  viz_06c_01_circuit_mi_trajectory_pythia-160m.png
  viz_06c_01_circuit_mi_trajectory_pythia-1b.png
  viz_06c_01_circuit_mi_trajectory_pythia-410m.png
  viz_06c_01_circuit_mi_trajectory_pythia-70m.png
  viz_06c_02_circuit_mi_all_models.